# Práctica 3.6 Redes Neuronales
## Universidad Nacional Autónoma de México - Facultad de Ingeniería
## Minería de Datos - Gpo. 03 - Semestre 2027-1
### Autor: Lee Robles Leonardo Alan

---

## **1. Propósito de la práctica**

Diseñar, entrenar y validar una herramienta basada en Redes Neuronales Convolucionales (CNN) para detectar y clasificar software malicioso a través de un mapeo del código binario del malware a una imagen bidimensional utilizando herramientas de *Machine Learning* y el dataset "Malimg".

---

In [ ]:
import os 
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

Image.MAX_IMAGE_PIXELS = None #Avoid Image.DecompressionBombError

## **2. Desarrollo de la práctica**


### 2.1 Análisis exploratorio (EDA) del dataset


#### 2.1.1 Escaneo del dataset y construcción del DataFrame de metadatos

In [ ]:
files_registers = []

for root, dirs, files in os.walk('malimg_dataset'):
    for file in files:
        
        attributes = {}

        if not file.endswith('.png'):
            continue

        full_path = os.path.join(root, file)

        attributes['path'] = full_path
        attributes['size_kb'] = os.path.getsize(full_path) / (1024)
        attributes['family'] = os.path.basename(root)

        img = Image.open(full_path)

        attributes['width'] = img.width
        attributes['height'] = img.height
        attributes['resolution'] = (img.width, img.height)
        attributes['mode'] = img.mode

        files_registers.append(attributes)

EDA_df = pd.DataFrame(files_registers)

print(EDA_df)    

##### 2.1.1.1 Análisis de balance de clases

Utilizamos la función `.value_counts()` de Pandas para obtener el conteo total de imágenes por familia.

In [ ]:
# Count the number of rows per family
counts = EDA_df.value_counts('family')

# Order the families by its size
fam_order = counts.index

# Create plot
plt.figure(figsize=(10,8))
ax = sns.countplot(data=EDA_df, y='family', order=fam_order)

# Calculate percentage
total_images = len(EDA_df)
percentages = [f'{(val / total_images) * 100:.1f}% ({val})' for val in counts]

# Assign labels
ax.bar_label(ax.containers[0], labels=percentages,padding=3)

plt.title('Balance de clases en familias de malware')
plt.xlabel('Número de imágenes')
plt.ylabel('Familia')

plt.tight_layout()
plt.show()

##### 2.1.1.2 Análisis de dimensiones, resoluciones y tamaños

El modelo solamente puede recibir entradas de tamaño fijo, por lo que si las imágenes del *dataset* son muy variadas en cuanto a resolución y tamaño, debemos tener en cuenta la re-escalación de la imagen, introduciendo pérdidas o ruido al modelo.

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=EDA_df, x='width', y='height', hue='family', alpha=0.6)

# To avoid the legend box to cover the plot
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left', borderaxespad=0.)

plt.title('Dispersión de resoluciones (ancho vs. alto)')
plt.tight_layout()
plt.show()

Esto nos indica que diferentes familias de Malware tienen diferentes resoluciones en las imágenes, lo que nos puede dar un indicio para clasificar. 

Ahora, para determinar hacia qué resolución normalizaremos todos las imágenes, observaremos la moda para el ancho y el alto. **Nota:** El ancho es una variable discreta, mientras que el alto es una variable continua.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))

# Width
sns.countplot(data=EDA_df, x='width', ax=axes[0], color='skyblue')
axes[0].set_title('Distribución del ancho (pixeles)')
axes[0].tick_params(axis='x', rotation=45)

# Height
sns.histplot(data=EDA_df[EDA_df['height'] <= 1024], x='height', bins=50, ax=axes[1], color='salmon')
axes[1].set_title('Distribución de alto (pixeles)')

max_patch = max(axes[1].patches, key=lambda p: p.get_height())

# Get the X mid point and the Y height peak
bin_center = max_patch.get_x() + (max_patch.get_width() / 2)
max_height = max_patch.get_height()

axes[1].annotate(
    f'{bin_center:.0f} px', 
    xy=(bin_center, max_height),
    xytext=(0, 5),
    textcoords='offset points',
    ha='center',
    va='bottom', 
    color='darkred'
)

plt.tight_layout()
plt.show()

In [ ]:
# Convert to MB
EDA_df['size_mb'] = EDA_df['size_kb'] / (1024)

# Calculate the average per family and sort ascendently
avg_sizes = EDA_df.groupby('family')['size_mb'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 8))

order_by_size = avg_sizes.index

sns.boxplot(data=EDA_df, y='family', x='size_mb', order=order_by_size, color='purple')

plt.title('Distribución del tamaño de archivo por familia')
plt.xlabel('Tamaño (MB)')
plt.ylabel('Familia')
plt.tight_layout()
plt.show()

##### 2.1.1.3 Matriz de subplots de imágenes reales

In [ ]:
# Define the families to show and the samples per family
families_to_show = EDA_df['family'].unique() #Add [:3] to limit only 3 families.
samples_per_family = 5

# Create the subplots
fig, axes = plt.subplots(
    nrows=len(families_to_show), 
    ncols=samples_per_family, 
    figsize=(10, 2.5 * len(families_to_show))
)

# Iterate per family and render its images
for row_idx, family_name in enumerate(families_to_show):
    # Get the paths of the 'n' images from a family
    family_paths = EDA_df[EDA_df['family'] == family_name]['path'].sample(samples_per_family).values
    
    for col_idx in range(samples_per_family):
        ax = axes[row_idx, col_idx] if len(families_to_show) > 1 else axes[col_idx]
        
        # Load and display the images
        img_path = family_paths[col_idx]
        img = Image.open(img_path)
        
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        
        # Family name
        if col_idx == 0:
            ax.set_title(family_name, loc='left', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

##### 2.1.1.4 Distribución de intensidad de píxeles

In [ ]:
families_to_show = EDA_df['family'].unique() #Add [:3] to limit only 3 families.
samples_per_family = 15

pixel_data = []

for row_idx, family_name in enumerate(families_to_show):

    family_paths = EDA_df[EDA_df['family'] == family_name]['path'].sample(samples_per_family).values
    
    for col_idx in range(samples_per_family):

        img_path = family_paths[col_idx]
        img = Image.open(img_path).convert('L')

        img_matrix = np.asarray(img)
        img_vector = img_matrix.flatten()
        img_vector_sample = np.random.choice(img_vector, size = min(len(img_vector), 10000))

        df_temp = pd.DataFrame(
            {
                'pixel_value':img_vector_sample,
                'family': family_name 
            }
        )

        pixel_data.append(df_temp)

px_df = pd.concat(pixel_data, ignore_index=True)

plt.figure(figsize=(10,8))
sns.kdeplot(data=px_df, x='pixel_value', hue='family',common_norm=False)

plt.title('Distribución de intensidad de pixeles por familia')
plt.xlim(0,255)
plt.tight_layout()
plt.show()


        

### 2.2 Preparación de los datos

#### 2.2.1 Separación de los datos

Se dividen los datos en 80% para entrenamiento, 10% para pruebas y 10% para validación

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    EDA_df['path'], 
    EDA_df['family'], 
    test_size=0.20, 
    stratify=EDA_df['family'], 
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=0.50, 
    stratify=y_temp, 
    random_state=42
)

print(f"Train samples: {len(X_train)} | Val samples: {len(X_val)} | Test samples: {len(X_test)}")

##### 2.2.1.1 Estrategia de balanceo

In [ ]:
def calc_weight(y_train):
    classes = np.unique(y_train)

    class_weights_vector = compute_class_weight(
        class_weight='balanced', 
        classes=classes, 
        y=y_train
    )

    class_weights_dict = dict(zip(classes, class_weights_vector))

    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
    idx_class_weights = {class_to_idx[cls]: weight for cls, weight in class_weights_dict.items()}

    print("Diccionario de pesos por clase:", idx_class_weights)
    
    return idx_class_weights

In [ ]:
pesos_diccionario = calc_weight(y_train)

#### 2.2.2 Transformación de los datos

Los datos pasan por un proceso de *augmentation* (crear nuevos datos a partir de los ya existentes), redimensionamiento (para que las entradas al CNN sean todas del mismo tamaño) y normalización

#### 2.2.2.1 Data augmentation

Provoca variaciones en las imágenes para aumentar la cardinalidad del dataset

In [ ]:
def data_augmentation(img_matrix, target_size=(256, 256)):

    # Horizontal flip with a 50% chance
    if np.random.rand() > 0.5:
        img_matrix = np.fliplr(img_matrix)

    # Brightness adjustmente (between 85% and 115%)
    if np.random.rand() > 0.5:
        factor = np.random.uniform(0.85, 1.15)
        img_matrix = np.clip(img_matrix.astype(np.float32) * factor, 0, 255).astype(np.uint8)

    # Slight crop (crops and resizes)
    if np.random.rand() > 0.5:
        h, w = img_matrix.shape[:2]
        crop_h, crop_w = int(h * 0.90), int(w * 0.90)
        
        top = (h - crop_h) // 2
        left = (w - crop_w) // 2
        
        cropped = img_matrix[top:top + crop_h, left:left + crop_w]
        
        # Resize to 256x256 resolution
        img_pil = Image.fromarray(cropped, mode='L')
        img_resized = img_pil.resize(target_size, resample=Image.NEAREST)
        img_matrix = np.asarray(img_resized)

    return img_matrix


##### 2.2.2.2 Estandarización y redimensionamiento

Se decidió tomar la resolución objetivo de 256x256 para todo el dataset, aplicando un redimensionamiento, en lugar de un *zero-padding*. Se redimensionará todo el dataset y se almacenará en disco. Se tomó esta decisión para no sobrecargar la RAM cada vez que se ejecute la CNN.

In [ ]:
def resize_image(img, target_size=(256, 256)):
    return img.resize(target_size, resample=Image.NEAREST)


##### 2.2.2.3 Normalización

En general, el dataset no presenta sesgos extremos en las intensidades de su píxeles, por lo que se aplicará una normalización min-max.

In [ ]:
def normalize_matrix(img_matrix):
    normalized = img_matrix.astype(np.float32) / 255.0
    return np.expand_dims(normalized, axis=-1)

#### 2.2.3 Pipeline de carga


In [ ]:
def load_pipeline(img_path, is_training=True, target_size=(256, 256)):
    
    img = Image.open(img_path).convert('L')
    img_resized = resize_image(img, target_size)
    img_matrix = np.asarray(img_resized)
    
    if is_training:
        img_matrix = data_augmentation(img_matrix, target_size)
        
    final_tensor = normalize_matrix(img_matrix)
    
    return final_tensor



### 2.3 Entrenamiento del modelo

### 2.4 Validación del modelo

### 2.5 Pruebas y *testing* del modelo

---

## **3. Resultados**

---

## **4. Conclusiones de la práctica**

### Referencias

- Nataraj, L., Yegneswaran, V., Porras, P., & Zhang, X. (2011). Malware images: Visualization and automatic classification. En Proceedings of the 8th International Symposium on Visualization for Cyber Security (VizSec '11) (pp. 1–7). Association for Computing Machinery. https://dl.acm.org/doi/10.1145/2016904.2016908
